#Imports
code reference to https://github.com/antoinekeller/tennis_shot_recognition/blob/master/SingleFrameShotClassifier.ipynb

In [ ]:
import tensorflow as tf
config = tf.compat.v1.ConfigProto()
config.gpu_options.allow_growth = True
session = tf.compat.v1.Session(config=config)

from tensorflow import keras
from tensorflow.keras import layers
physical_devices = tf.config.experimental.list_physical_devices('GPU')

from tensorflow import keras
from tensorflow.keras import layers

from keras.models import Sequential
from keras.layers import Convolution2D, Dropout, Dense
from keras.layers import BatchNormalization
from keras.layers import MaxPooling2D
from keras.layers import Flatten
from keras.layers import LeakyReLU
from keras.callbacks import ModelCheckpoint

from sklearn import preprocessing
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import pandas as pd
import seaborn as sn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import os

# Load labeled dataset of single frame human pose

In [ ]:
X=[]
y=[]
X_test = []
y_test = []

# download dataset from drive
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/dataset /content/

folders = ["alcaraz", "dimitrov_alcaraz", "dimitrov_thiem", "roland", "djoko_sock", "federer", "federer_volley", "alcaraz_volley", "murray", "fokina", "djokovic"]

DATASET_PATH = "/content/dataset"
for folder in folders:
    if not os.path.exists(f"dataset/{folder}/shots/"):
        print(f"dataset/{folder}/shots/ doesnt exist")
        continue

    print(f"Loading shots from {DATASET_PATH}{folder}/shots/")

    for shot_csv in tqdm(sorted(os.listdir(f'{DATASET_PATH}/{folder}/shots/'))):

      # # only include file with serving and neutral  frames
      # if not shot_csv.startwith("serve") or not shot_csv.startswith("neutral"):
      #   continue

      data = pd.read_csv(os.path.join(f'{DATASET_PATH}/{folder}/shots/', shot_csv))


      # split the train/test set 80/20
      data_train = data[:int(0.8*len(data))]
      data_test = data[int(0.8*len(data)):]

      #print(f"{len(data_train)} for training and {len(data_test)} for test")

      #print(list(data_train.loc[:, data.columns != 'shot'].to_numpy()))
      features_train = list(data_train.loc[:, data.columns != 'shot'].to_numpy())
      features_test = list(data_test.loc[:, data.columns != 'shot'].to_numpy())

      #print(features_train)

      X.extend(features_train)
      y.extend(data_train["shot"].to_numpy().flatten())

      X_test.extend(features_test)
      y_test.extend(data_test["shot"].to_numpy().flatten())



X = np.stack(X, axis=0)

y = np.array(y)
X = np.array(X)

y_test = np.array(y_test)

print(f"Loaded {len(y)} shots for training")
print(f"Loaded {len(y_test)} shots for test")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading shots from /content/datasetalcaraz/shots/


100%|██████████| 129/129 [00:00<00:00, 399.46it/s]


dataset/dimitrov_alcaraz/shots/ doesnt exist
Loading shots from /content/datasetdimitrov_thiem/shots/


100%|██████████| 143/143 [00:00<00:00, 429.58it/s]


Loading shots from /content/datasetroland/shots/


...

Loading shots from /content/datasetfokina/shots/


100%|██████████| 51/51 [00:00<00:00, 420.49it/s]


Loading shots from /content/datasetdjokovic/shots/


100%|██████████| 23/23 [00:00<00:00, 411.12it/s]


Loaded 27264 shots for training
Loaded 6816 shots for test


# Make training validation dataset

In [ ]:
# Split the data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.33, shuffle= True)



print(f"Shape of train features : {X_train[0].shape}")
print(f"Shape of val features : {X_val[0].shape}")


print("Total categories: ", len(np.unique(y_train)))
print("Total categories: ", len(np.unique(y_val)))

nb_cat = len(np.unique(y_train))

Shape of train features : (26,)
Shape of val features : (26,)
Total categories:  6
Total categories:  6


In [ ]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()


y_train = le.fit_transform(y_train)
y_val = le.fit_transform(y_val)
y_test = le.fit_transform(y_test)

y_train = tf.keras.utils.to_categorical(y_train, num_classes=nb_cat)
y_val = tf.keras.utils.to_categorical(y_val, num_classes=nb_cat)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=nb_cat)

y_train = np.array(y_train)
X_train = np.array(X_train)

y_val = np.array(y_val)
X_val = np.array(X_val)

y_test = np.array(y_test)
X_test = np.array(X_test)

In [ ]:
print(list(le.classes_))

[np.str_('backhand'), np.str_('backhand-volley'), np.str_('forehand'), np.str_('forehand-volley'), np.str_('neutral'), np.str_('serve')]


In [ ]:
print("X_train Shape: ", X_train.shape)
print("X_val Shape: ", X_val.shape)
print("X_test Shape: ", X_test.shape)
print("y_train Shape: ", y_train.shape)
print("y_val Shape: ", y_val.shape)
print("y_test Shape: ", y_test.shape)

X_train Shape:  (18266, 26)
X_val Shape:  (8998, 26)
X_test Shape:  (6816, 26)
y_train Shape:  (18266, 6)
y_val Shape:  (8998, 6)
y_test Shape:  (6816, 6)


In [ ]:
m1=Sequential()
m1.add(Dense(units=16,activation = 'relu', input_shape=(26,)))
#m1.add(Dropout(0.5))
m1.add(Dense(units=8,activation = 'relu', input_shape=(26,)))
#m1.add(Dropout(0.5))
m1.add(Dense(units = 8, activation = 'relu'))
m1.add(Dense(units = nb_cat, activation = 'softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
m1.compile(optimizer='adam', loss = 'categorical_crossentropy',metrics = ['accuracy'])
m1.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │            54 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 694 (2.71 KB)

 Trainable params: 694 (2.71 KB)

 Non-trainable params: 0 (0.00 B)

# Train

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint
filepath = "weights.keras"
checkpointer = ModelCheckpoint(filepath=filepath, verbose=False, save_best_only=True)
hist = m1.fit(X_train, y_train,
               validation_data=(X_val, y_val),
               batch_size = 32,
                epochs=300,
                verbose = 1,
                callbacks=[checkpointer])

loss, accuracy = m1.evaluate(X_val, y_val)
print(f"Accuracy on validation dataset = {accuracy}")

Epoch 1/300
571/571 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.4362 - loss: 1.4244 - val_accuracy: 0.4637 - val_loss: 1.2916
Epoch 2/300
571/571 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6210 - loss: 1.0508 - val_accuracy: 0.7156 - val_loss: 0.8956
Epoch 3/300
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7102 - loss: 0.8552 - val_accuracy: 0.7233 - val_loss: 0.8154
Epoch 4/300
571/571 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7262 - loss: 0.7920 - val_accuracy: 0.7242 - val_loss: 0.7786
Epoch 5/300
571/571 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7379 - loss: 0.7575 - val_accuracy: 0.7483 - val_loss: 0.7379
Epoch 6/300
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7467 - loss: 0.7303 - val_accuracy: 0.7546 - val_loss: 0.7057
Epoch 7/300
571/571 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7544 - loss: 0.7076 - val_accuracy: 0.7647 - val_loss: 0.6891
Epoch 8/300
571/571 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7602 - loss: 0.6909 - val_accu

# Model Analysis